# NeuroVista — MONAI BraTS inference in Google Colab

**Research and education only — not for diagnosis.** Run the numbered stages in order. Each stage checks its prerequisites and stops with a corrective instruction. NeuroVista remains **Local inference not connected**; downloaded files are imported separately as **Cloud-generated AI prediction imported**. There is no live Colab connection. Use only de-identified research data you are authorized to upload.

## 1. Check Python and runtime
This adapted environment requires Python 3.12. Select a GPU runtime before continuing.

In [ ]:
import os, platform, shutil, sys
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f'Expected Python 3.12; received {platform.python_version()}. Start a fresh compatible Colab runtime.')
disk = shutil.disk_usage('/content')
print({'python': platform.python_version(), 'runtime_pid': os.getpid(), 'free_disk_gib': round(disk.free / 1024**3, 2)})
print('Stage 1 passed. Run Stage 2 once.')

## 2. Install the pinned Python 3.12 stack — run once
The original v0.5.4 metadata used NumPy 1.24.4. The adapted Python 3.12 environment uses NumPy 1.26.4 while preserving the architecture, weights and inference contract. Checked subprocesses ensure installation cannot report success after a failed command.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
if sys.version_info[:2] != (3, 12): raise RuntimeError('Run Stage 1 in a Python 3.12 runtime.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--index-url', 'https://download.pytorch.org/whl/cu121', 'torch==2.4.0', 'torchvision==0.19.0', 'torchaudio==2.4.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'monai==1.4.0', 'numpy==1.26.4', 'nibabel==5.2.1', 'pytorch-ignite==0.4.11', 'scikit-learn==1.5.1', 'tensorboard==2.17.0', 'huggingface_hub==0.27.1', 'psutil==6.1.1', 'packaging==24.2'], check=True)
Path('/content/neurovista_restart_required.json').write_text(json.dumps({'installer_pid': os.getpid()}), encoding='utf-8')
print('INSTALLATION SUCCEEDED.')
print('REQUIRED ONCE: choose Runtime > Restart session now. Do not rerun Stage 2 after restart; continue at Stage 3.')

## REQUIRED RESTART — exactly once
Choose **Runtime → Restart session** now, then continue with Stage 3. Stage 3 detects if the restart was skipped.

## 3. Import everything and verify versions, RAM and GPU
This cell recreates all imports, model constants, channel order and hardware state after restart.

In [ ]:
import hashlib, importlib.metadata, json, os, platform, shutil, tarfile, time, urllib.request, zipfile
from datetime import datetime, timezone
from pathlib import Path
import nibabel as nib
import numpy as np
import psutil, torch, monai
from huggingface_hub import hf_hub_download
from monai.inferers import sliding_window_inference
from monai.networks.nets import SegResNet
from monai.transforms import NormalizeIntensity
from packaging.version import Version
marker = Path('/content/neurovista_restart_required.json')
if not marker.is_file(): raise RuntimeError('Installation marker missing. Run Stages 1 and 2, then restart once.')
if json.loads(marker.read_text())['installer_pid'] == os.getpid(): raise RuntimeError('Restart not detected. Choose Runtime > Restart session, then rerun Stage 3.')
ORIGINAL_MODEL_ENVIRONMENT = {'torch':'2.4.0','monai':'1.4.0','numpy':'1.24.4','nibabel':'5.2.1','pytorch-ignite':'0.4.11','scikit-learn':'1.5.1','tensorboard':'2.17.0'}
EXPECTED_ADAPTED_STACK = {'torch':'2.4.0','torchvision':'0.19.0','torchaudio':'2.4.0','monai':'1.4.0','numpy':'1.26.4','nibabel':'5.2.1','pytorch-ignite':'0.4.11','scikit-learn':'1.5.1','tensorboard':'2.17.0','huggingface-hub':'0.27.1','psutil':'6.1.1','packaging':'24.2'}
resolved = {name: importlib.metadata.version(name) for name in EXPECTED_ADAPTED_STACK}
mismatches = {name:{'expected':expected,'actual':resolved[name]} for name,expected in EXPECTED_ADAPTED_STACK.items() if Version(resolved[name]).base_version != expected}
if mismatches: raise RuntimeError(f'Pinned stack mismatch: {mismatches}. Rerun Stage 2 and restart once.')
if not torch.cuda.is_available(): raise RuntimeError('CUDA unavailable. Choose Runtime > Change runtime type > GPU, then rerun Stage 3.')
ram = psutil.virtual_memory()
HARDWARE = {'ram_total_gib':round(ram.total/1024**3,3),'ram_available_gib':round(ram.available/1024**3,3),'cuda_available':True,'gpu':torch.cuda.get_device_name(0),'gpu_vram_gib':round(torch.cuda.get_device_properties(0).total_memory/1024**3,3),'torch_cuda_build':torch.version.cuda}
WORK=Path('/content/neurovista_inference'); CACHE=Path('/content/neurovista_cache'); WORK.mkdir(exist_ok=True); CACHE.mkdir(exist_ok=True)
CHANNEL_ORDER=['t1c','t1','t2','flair']; REFERENCE_MODALITY='t1c'
MODEL_REPO='MONAI/brats_mri_segmentation'; MODEL_VERSION='0.5.4'; MODEL_FILE='models/model.pt'
MODEL_SOURCE=f'https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_VERSION}/{MODEL_FILE}'; EXPECTED_WEIGHT_SHA256='860ccb3f1c21c99d0410ad8a1ac4ef6b8fab60cec0a503b0ba42675741a750ae'
RUNTIME_READY=True
print(json.dumps({'original_model_environment':ORIGINAL_MODEL_ENVIRONMENT,'adapted_python312_environment':{'python':platform.python_version(),**resolved},'hardware':HARDWARE},indent=2))

## 4. Download and verify official weights
The versioned MONAI file is reused from cache and its published SHA-256 is always verified.

In [ ]:
if not globals().get('RUNTIME_READY'): raise RuntimeError('Run Stage 3 first.')
weight_path=Path(hf_hub_download(repo_id=MODEL_REPO,filename=MODEL_FILE,revision=MODEL_VERSION,cache_dir=CACHE/'huggingface'))
weight_sha256=hashlib.sha256(weight_path.read_bytes()).hexdigest()
if weight_sha256 != EXPECTED_WEIGHT_SHA256: raise RuntimeError(f'Weight checksum mismatch: {weight_sha256}. Remove the corrupt cache and rerun Stage 4.')
WEIGHTS_READY=True
print({'source':MODEL_SOURCE,'version':MODEL_VERSION,'sha256':weight_sha256,'cached_path':str(weight_path)})

## 5. Choose and prepare one data source
Use `msd_demo` for official MSD Task01 BRATS_457, or `upload` for four compatible research volumes. The MSD branch must download the approximately 7.61 GB archive even though it extracts one case, but reuses a checksum-verified cache. The model channel order is always T1c, T1, T2, FLAIR; MSD source order is 0=FLAIR, 1=T1, 2=T1c/T1gd, 3=T2.

In [ ]:
if not globals().get('RUNTIME_READY'): raise RuntimeError('Run Stage 3 first.')
DATA_SOURCE='msd_demo'  # exactly 'msd_demo' or 'upload'
USER_INPUT_NAMES={'t1c':'','t1':'','t2':'','flair':''}
USER_DATASET_ATTRIBUTION={'dataset':'','source':'','licence':''}
USER_REFERENCE_NAME=''  # optional expert-reference .nii or .nii.gz filename
ALLOW_USER_T1C_REDISTRIBUTION_IN_PACKAGE=False
if DATA_SOURCE not in {'msd_demo','upload'}: raise ValueError("Choose DATA_SOURCE='msd_demo' or 'upload'.")
INPUTS=None; REFERENCE_MASK=None; DATASET_ATTRIBUTION=None
print(f'Selected {DATA_SOURCE}. Run the matching preparation cell only.')

### 5A. Official MSD BRATS_457 demonstration
Run only for `msd_demo`. It checks disk, verifies MD5, safely extracts only the image, expert mask and metadata, and preserves `.nii.gz` compression.

In [ ]:
if DATA_SOURCE!='msd_demo':
    print('Skipped 5A. Run 5B.')
else:
    archive_path=CACHE/'Task01_BrainTumour.tar'; archive_url='https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar'; expected_md5='240a19d752f0d9e9101544901065d872'
    def digest_file(path,algorithm='sha256'):
        d=hashlib.new(algorithm)
        with open(path,'rb') as stream:
            for chunk in iter(lambda:stream.read(8*1024*1024),b''): d.update(chunk)
        return d.hexdigest()
    cached_ok=archive_path.is_file() and digest_file(archive_path,'md5')==expected_md5
    if not cached_ok:
        free=shutil.disk_usage('/content').free
        if free < 9*1024**3: raise RuntimeError(f'Need at least 9 GiB free before the ~7.61 GB download; only {free/1024**3:.2f} GiB is free.')
        if archive_path.exists(): archive_path.unlink()
        print('Downloading official archive once; progress follows.')
        def report(blocks,size,total):
            done=min(blocks*size,total) if total>0 else blocks*size; print(f'\r{done/1024**3:.2f} / {total/1024**3:.2f} GiB',end='')
        urllib.request.urlretrieve(archive_url,archive_path,reporthook=report); print()
    actual_md5=digest_file(archive_path,'md5')
    if actual_md5!=expected_md5: raise RuntimeError(f'Archive MD5 mismatch: {actual_md5}. Delete it and rerun; do not infer from it.')
    selected=WORK/'msd_brats_457'; selected.mkdir(parents=True,exist_ok=True)
    members={'image':'Task01_BrainTumour/imagesTr/BRATS_457.nii.gz','reference':'Task01_BrainTumour/labelsTr/BRATS_457.nii.gz','metadata':'Task01_BrainTumour/dataset.json'}
    targets={'image':selected/'BRATS_457_4d.nii.gz','reference':selected/'BRATS_457_reference.nii.gz','metadata':selected/'dataset.json'}
    with tarfile.open(archive_path,'r') as archive:
        names=set(archive.getnames()); missing=set(members.values())-names
        if missing: raise RuntimeError(f'Verified archive is missing {sorted(missing)}')
        for key,name in members.items():
            member=archive.getmember(name)
            if not member.isfile() or '..' in Path(name).parts: raise RuntimeError(f'Unsafe member: {name}')
            if not targets[key].is_file():
                with archive.extractfile(member) as src, open(targets[key],'wb') as dst: shutil.copyfileobj(src,dst)
    image=nib.load(str(targets['image'])); data=np.asarray(image.dataobj)
    if data.ndim!=4 or data.shape[-1]!=4: raise RuntimeError(f'Expected 4D/4-channel image, got {data.shape}')
    split=WORK/'BRATS_457_split'; split.mkdir(exist_ok=True); source_map={'flair':0,'t1':1,'t1c':2,'t2':3}; INPUTS={}
    for modality,index in source_map.items():
        output=split/f'BRATS_457_{modality}.nii.gz'
        if not output.is_file(): nib.save(nib.Nifti1Image(data[...,index],image.affine,image.header.copy()),output)
        INPUTS[modality]=str(output)
    REFERENCE_MASK=targets['reference']
    DATASET_ATTRIBUTION={'dataset':'Medical Segmentation Decathlon Task01 BrainTumour','case':'BRATS_457','source':archive_url,'licence':'CC BY-SA 4.0','archive_md5':actual_md5,'source_channel_mapping':{'0':'FLAIR','1':'T1','2':'T1c/T1gd','3':'T2'}}
    DATA_READY=True
    print(json.dumps({'inputs':INPUTS,'reference_mask':str(REFERENCE_MASK),'attribution':DATASET_ATTRIBUTION,'cache_reused':cached_ok},indent=2))

### 5B. User-supplied volumes
Run only for `upload`. Fill all filenames and attribution fields in Stage 5 first. This branch never overwrites prepared MSD inputs with placeholders.

In [ ]:
if DATA_SOURCE!='upload':
    print('Skipped 5B. Keep 5A inputs.')
else:
    from google.colab import files
    if any(not USER_INPUT_NAMES.get(name) for name in CHANNEL_ORDER): raise ValueError(f'Fill USER_INPUT_NAMES for {CHANNEL_ORDER}.')
    if any(not USER_DATASET_ATTRIBUTION.get(name) for name in ('dataset','source','licence')): raise ValueError('Fill dataset, source and licence attribution before upload.')
    uploaded=files.upload(); requested=list(USER_INPUT_NAMES.values())+([USER_REFERENCE_NAME] if USER_REFERENCE_NAME else []); missing=[name for name in requested if name not in uploaded and not Path('/content',name).is_file()]
    if missing: raise FileNotFoundError(f'Missing {missing}. Upload those exact files and rerun 5B.')
    INPUTS={modality:str(Path('/content',name)) for modality,name in USER_INPUT_NAMES.items()}
    REFERENCE_MASK=Path('/content',USER_REFERENCE_NAME) if USER_REFERENCE_NAME else None; DATASET_ATTRIBUTION={**USER_DATASET_ATTRIBUTION,'case':'user-supplied de-identified research case'}; DATA_READY=True
    print(json.dumps({'inputs':INPUTS,'attribution':DATASET_ATTRIBUTION},indent=2))

## 6. Validate geometry and preprocess
Requires finite aligned 3D volumes, 1 mm isotropic spacing and millimetre units. Applies release-exact nonzero channel-wise normalization; no silent registration, reorientation or resampling.

In [ ]:
if not globals().get('DATA_READY') or not isinstance(INPUTS,dict): raise RuntimeError('Complete one Stage 5 branch first.')
def sha256_file(path):
    d=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''): d.update(chunk)
    return d.hexdigest()
images={}; arrays={}; input_manifest={}
for modality in CHANNEL_ORDER:
    path=Path(INPUTS[modality])
    if not path.is_file() or not (path.name.lower().endswith('.nii') or path.name.lower().endswith('.nii.gz')): raise FileNotFoundError(f'{modality}: correct Stage 5; missing NIfTI {path}')
    image=nib.load(str(path)); values=np.asarray(image.dataobj); affine=np.asarray(image.affine,dtype=float); spacing=np.asarray(nib.affines.voxel_sizes(affine))
    if values.ndim!=3 or not np.issubdtype(values.dtype,np.number) or not np.isfinite(values).all(): raise ValueError(f'{modality}: requires one finite numeric 3D volume')
    if not np.isfinite(affine).all() or abs(np.linalg.det(affine[:3,:3]))<1e-12: raise ValueError(f'{modality}: invalid affine')
    if not np.allclose(spacing,(1,1,1),rtol=0,atol=1e-3): raise ValueError(f'{modality}: expected 1 mm spacing, got {spacing.tolist()}')
    if image.header.get_xyzt_units()[0]!='mm': raise ValueError(f'{modality}: spatial unit must be mm')
    images[modality]=image; arrays[modality]=values.astype(np.float32,copy=False); input_manifest[modality]={'sha256':sha256_file(path),'shape':list(values.shape),'spacing_mm':spacing.tolist(),'axis_codes':list(nib.aff2axcodes(affine)),'affine':affine.tolist()}
reference=images[REFERENCE_MODALITY]
for modality in CHANNEL_ORDER[1:]:
    if images[modality].shape!=reference.shape: raise ValueError(f'{modality}: shape mismatch; supply co-registered inputs')
    if not np.allclose(images[modality].affine,reference.affine,rtol=1e-5,atol=1e-4): raise ValueError(f'{modality}: affine mismatch; automatic alignment is disabled')
stacked=np.stack([arrays[name] for name in CHANNEL_ORDER],axis=0)
normalized=NormalizeIntensity(nonzero=True,channel_wise=True)(stacked)
model_input=torch.as_tensor(np.asarray(normalized),dtype=torch.float32).unsqueeze(0); PREPROCESSING_READY=True
print(json.dumps({'model_channel_order':CHANNEL_ORDER,'tensor_shape':list(model_input.shape),'inputs':input_manifest},indent=2))

## 7. Run inference
Release-exact SegResNet; ROI 240×240×160; batch 1; overlap 0.5; sigmoid; threshold 0.5. Later stages refuse to reconstruct missing results.

In [ ]:
if not globals().get('WEIGHTS_READY'): raise RuntimeError('Complete Stage 4.')
if not globals().get('PREPROCESSING_READY'): raise RuntimeError('Complete Stage 6.')
device=torch.device('cuda:0')
model=SegResNet(spatial_dims=3,init_filters=16,in_channels=4,out_channels=3,dropout_prob=0.2,blocks_down=(1,2,2,4),blocks_up=(1,1,1)).to(device)
checkpoint=torch.load(weight_path,map_location='cpu',weights_only=True); state_dict=checkpoint.get('model',checkpoint) if isinstance(checkpoint,dict) else checkpoint
model.load_state_dict(state_dict,strict=True); model.eval()
def predictor(window):
    with torch.autocast(device_type='cuda',dtype=torch.float16): return model(window)
torch.cuda.reset_peak_memory_stats(); started=time.perf_counter()
with torch.inference_mode(): logits=sliding_window_inference(model_input.to(device),roi_size=(240,240,160),sw_batch_size=1,predictor=predictor,overlap=0.5,device=torch.device('cpu'),sw_device=device)
elapsed_seconds=time.perf_counter()-started; channels=(torch.sigmoid(logits)[0]>=0.5).cpu().numpy()
if channels.shape!=(3,*reference.shape): raise RuntimeError(f'Unexpected output {channels.shape}; no valid inference result exists.')
peak_gpu_memory_gib=torch.cuda.max_memory_allocated()/1024**3; INFERENCE_COMPLETE=True
print({'inference_complete':True,'elapsed_seconds':elapsed_seconds,'peak_gpu_memory_gib':peak_gpu_memory_gib,'foreground_voxels_tc_wt_et':[int(v.sum()) for v in channels]})

## 8. Export prediction and provenance
Label precedence is ET→4, otherwise TC→1, otherwise WT→2, otherwise 0. Shape and affine are rechecked against T1c.

In [ ]:
if not globals().get('INFERENCE_COMPLETE') or 'channels' not in globals(): raise RuntimeError('Run Stage 7. Never fabricate or reconstruct missing output.')
tc,wt,et=channels; label_map=np.zeros(reference.shape,dtype=np.uint8); label_map[wt]=2; label_map[tc]=1; label_map[et]=4
result_dir=WORK/'results'; result_dir.mkdir(parents=True,exist_ok=True); mask_path=result_dir/'neurovista_ai_prediction.nii.gz'
header=reference.header.copy(); header.set_data_dtype(np.uint8); nib.save(nib.Nifti1Image(label_map,reference.affine.copy(),header=header),mask_path)
reloaded=nib.load(str(mask_path))
if reloaded.shape!=reference.shape or not np.allclose(reloaded.affine,reference.affine,rtol=0,atol=1e-6): raise RuntimeError('Export geometry failed; do not report this result.')
provenance={'application':'NeuroVista Colab inference','research_only':True,'completed_at_utc':datetime.now(timezone.utc).isoformat(),'model':{'name':'brats_mri_segmentation','version':MODEL_VERSION,'architecture':'SegResNet','source':MODEL_SOURCE,'weight_sha256':weight_sha256,'licence':'Apache-2.0'},'software':{'original_model_environment':ORIGINAL_MODEL_ENVIRONMENT,'adapted_python312_environment':{'python':platform.python_version(),**resolved},'adaptation':'NumPy 1.24.4 replaced by 1.26.4 for Python 3.12; model contract unchanged'},'hardware':HARDWARE,'dataset':DATASET_ATTRIBUTION,'input_contract':{'channel_order':CHANNEL_ORDER,'reference':REFERENCE_MODALITY,'inputs':input_manifest},'preprocessing':{'spatial_transform':'none; aligned 1 mm geometry required','intensity':'NormalizeIntensity(nonzero=True, channel_wise=True)'},'inference':{'roi_size':[240,240,160],'overlap':0.5,'activation':'sigmoid','threshold':0.5,'elapsed_seconds':elapsed_seconds,'peak_gpu_memory_gib':peak_gpu_memory_gib},'output':{'filename':mask_path.name,'sha256':sha256_file(mask_path),'shape':list(reloaded.shape),'affine':reloaded.affine.tolist(),'labels':{'0':'background','1':'tumour-core remainder','2':'whole-tumour-only remainder','4':'enhancing tumour'}},'limitations':['Not diagnostic accuracy','Expert review required','Possible training-subject overlap may be unresolved']}
provenance_path=result_dir/'neurovista_ai_prediction_provenance.json'; provenance_path.write_text(json.dumps(provenance,indent=2)+'\n',encoding='utf-8')
attribution_path=result_dir/'DATASET_ATTRIBUTION.json'; attribution_path.write_text(json.dumps(DATASET_ATTRIBUTION,indent=2)+'\n',encoding='utf-8'); EXPORT_COMPLETE=True
print({'prediction':str(mask_path),'sha256':sha256_file(mask_path),'geometry_verified':True,'provenance':str(provenance_path)})

## 9. Optionally compare with an expert reference
MSD reference regions: WT={1,2,3}, TC={2,3}, ET={3}. Prediction regions: WT={1,2,4}, TC={1,4}, ET={4}. Both empty is not applicable; one empty is Dice 0. This remains a single-case comparison, not independent validation.

In [ ]:
if not globals().get('EXPORT_COMPLETE'): raise RuntimeError('Complete Stage 8.')
comparison=None
if REFERENCE_MASK is None or not Path(REFERENCE_MASK).is_file():
    print('No expert reference available; comparison skipped without inventing metrics.')
else:
    ref_image=nib.load(str(REFERENCE_MASK)); ref=np.asarray(ref_image.dataobj); pred=np.asarray(reloaded.dataobj)
    if ref_image.shape!=reloaded.shape or not np.allclose(ref_image.affine,reloaded.affine,rtol=0,atol=1e-6): raise ValueError('Reference/prediction geometry mismatch; comparison refused.')
    if not np.isfinite(ref).all() or not np.allclose(ref,np.rint(ref)): raise ValueError('Reference labels must be finite integers.')
    ref=np.rint(ref).astype(np.int16); pred=np.rint(pred).astype(np.int16)
    if not set(np.unique(ref)).issubset({0,1,2,3}): raise ValueError(f'Unsupported reference labels {np.unique(ref).tolist()}')
    if not set(np.unique(pred)).issubset({0,1,2,4}): raise ValueError(f'Unsupported prediction labels {np.unique(pred).tolist()}')
    mappings={'whole_tumour':({1,2,3},{1,2,4}),'tumour_core':({2,3},{1,4}),'enhancing_tumour':({3},{4})}; regions={}
    for name,(ra,pa) in mappings.items():
        a=np.isin(ref,list(ra)); b=np.isin(pred,list(pa)); na=int(a.sum()); nb=int(b.sum()); inter=int(np.logical_and(a,b).sum())
        dice,status=(None,'not_applicable_both_empty') if na==0 and nb==0 else ((0.0,'one_empty') if na==0 or nb==0 else (2.0*inter/(na+nb),'calculated'))
        regions[name]={'dice':dice,'status':status,'reference_voxels':na,'prediction_voxels':nb,'intersection_voxels':inter}
    comparison={'scope':'single_case','independent_validation':False,'possible_training_subject_overlap':'unresolved','empty_region_policy':'both empty => null/not applicable; one empty => 0','regions':regions}
    comparison_path=result_dir/'expert_comparison.json'; comparison_path.write_text(json.dumps(comparison,indent=2)+'\n',encoding='utf-8'); print(json.dumps(comparison,indent=2))

## 10. Package and download
Includes prediction, provenance, attribution, matching compressed T1c, and the compressed reference when available and redistribution is permitted. Official MSD files are accompanied by CC BY-SA 4.0 attribution; user data requires the explicit redistribution flag.

In [ ]:
if not globals().get('EXPORT_COMPLETE'): raise RuntimeError('Complete Stage 8; partial packages are refused.')
from google.colab import files
package_files=[mask_path,provenance_path,attribution_path]; may_include=DATA_SOURCE=='msd_demo' or ALLOW_USER_T1C_REDISTRIBUTION_IN_PACKAGE
if may_include:
    t1c_copy=result_dir/Path(INPUTS['t1c']).name
    if not t1c_copy.is_file(): shutil.copy2(INPUTS['t1c'],t1c_copy)
    package_files.append(t1c_copy)
    if REFERENCE_MASK is not None and Path(REFERENCE_MASK).is_file():
        ref_copy=result_dir/('BRATS_457_reference.nii.gz' if DATA_SOURCE=='msd_demo' else Path(REFERENCE_MASK).name)
        if not ref_copy.is_file(): shutil.copy2(REFERENCE_MASK,ref_copy)
        package_files.append(ref_copy)
else: print('T1c/reference omitted because redistribution permission was not confirmed.')
if comparison is not None: package_files.append(comparison_path)
readme=result_dir/'README_IMPORT.txt'; readme.write_text('NeuroVista remains Local inference not connected. Load the matching T1c, choose AI prediction import, and import neurovista_ai_prediction.nii.gz as Cloud-generated AI prediction imported. Research only; see provenance and attribution.\n',encoding='utf-8'); package_files.append(readme)
package_path=WORK/'neurovista_prediction_package.zip'
with zipfile.ZipFile(package_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for item in package_files:
        if not Path(item).is_file(): raise FileNotFoundError(f'Package prerequisite missing: {item}')
        archive.write(item,arcname=Path(item).name)
PACKAGE_COMPLETE=True
print(json.dumps({'package':str(package_path),'sha256':sha256_file(package_path),'contents':[Path(p).name for p in package_files]},indent=2))
files.download(str(package_path)); print('Confirm the ZIP opens locally before cleanup.')

## 11. Cleanup only after download confirmation
Set `DOWNLOAD_CONFIRMED=True` only after opening the ZIP locally. Keep the verified cache to avoid an unnecessary repeat dataset download, or explicitly delete it.

In [ ]:
DOWNLOAD_CONFIRMED=False
DELETE_VERIFIED_CACHE_TOO=False
if not DOWNLOAD_CONFIRMED:
    print('Cleanup not run. Confirm the ZIP opens locally, then set DOWNLOAD_CONFIRMED=True.')
else:
    if not globals().get('PACKAGE_COMPLETE'): raise RuntimeError('No complete package was created in this session; cleanup refused.')
    shutil.rmtree(WORK,ignore_errors=True)
    if DELETE_VERIFIED_CACHE_TOO: shutil.rmtree(CACHE,ignore_errors=True)
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('Working files removed. Cache removed:',DELETE_VERIFIED_CACHE_TOO)
    print('Now use Runtime > Disconnect and delete runtime. Provider retention follows Google Colab policy.')